# Plot BMR for each mortality outcome

Using baseline period 1990-2009.

In [ ]:
import os
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import SymLogNorm
from matplotlib.ticker import FuncFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from utils.utils import autosize_figure, land_filter, create_global_country_map

In [ ]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
SAVE_DIR = "/glade/u/home/awells/air_quality_project/plotting/bmr/"

In [ ]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

In [ ]:
# === Baseline period ===
dates = "1990-2009"

In [ ]:
def plot_bmr(da, health_VAR, dates):
    # Create figure
    fig = plt.figure(figsize=autosize_figure(1, 1))
    gs = GridSpec(2, 1, height_ratios=[15, 1])  # rows, columns

    # Map projection and display
    projection = ccrs.Robinson()
    crs = ccrs.PlateCarree()

    country_borders = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none')

    # Use log scale for normalization
    vmax = da.max().item()
    cmap = plt.get_cmap("magma")

    # Define a linear threshold region around zero
    linthresh = 0.01

    norm = SymLogNorm(linthresh=linthresh, vmin=0, vmax=vmax, base=10)

    # First panel
    ax = fig.add_subplot(gs[0, 0], projection=projection, frameon=True)
    cb = da.plot(
        transform=crs,
        add_colorbar=False,
        cmap=cmap,
        norm=norm,
        subplot_kws={'projection': projection}
    )
    ax.coastlines(resolution="50m", linewidth=0.75)
    ax.add_feature(country_borders, edgecolor='k', linewidth=0.75)
    plt.title(f"Baseline Mortality Rate for {health_VAR}\n {dates} av.", fontsize=16)

    # First colorbar
    cax = fig.add_subplot(gs[1, 0])
    col_bar = plt.colorbar(cb, cax=cax, orientation='horizontal')
    col_bar.set_label("BMR per 100,000", fontsize=13)

    # Ticks
    formatter = FuncFormatter(lambda v, _: f"{v:g}")
    col_bar.ax.xaxis.set_major_formatter(formatter)

    plt.tight_layout()

    out_file = f"GBD_BMR_mean_{health_VAR}_{dates}.png"
    out_path = os.path.join(SAVE_DIR, out_file)
    plt.savefig(out_path)
    return

In [ ]:
# === Main loop ===

for health_VAR in health_vars:
    bmr_file = f"GBD_BMR_Country_{health_VAR}_newlabels_{dates}.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    bmr = xr.open_dataarray(bmr_path).sel(quantile="mean")  # central estimate

    # Project onto a 0.1ºx0.1º map
    bmr_map = create_global_country_map(bmr)

    # To be in units "per 100,000" and mask ocean
    bmr_mean = land_filter(bmr_map * 100000)

    plot_bmr(bmr_mean, health_VAR, dates)